# Analytics Run Evidence

Confirms that the BQ1 and BQ2 Analytics tables in `05_analytics` are correct, consistent with Gold, and safe to use for downstream analysis.

This notebook is **read-only**. It reads the Analytics tables, `04_gold.fact_province_year` and `06_data_quality.dq_results`. It does not write any table.

**Already checked in `01_build_analytics` (not repeated here):**
- Grain, row counts and duplicate keys (checks D1–D9, E1–E3, F1–F2, G1–G2)
- Eligibility and classification rules, including no metrics for missing or zero baselines and non-comparable years (E4–E6, F3–F6, G3–G12)
- Analytics values matching Gold (H1–H8)

These results are saved to `dq_results` and are **reported** in Section 1, not recalculated.

**Added in this notebook:**
- **Section 2:** selected units compared against values calculated directly from the **source files** (DOT PDFs, PSA tables, DS_18), which `01_build_analytics` does not do
- **Section 3:** exclusions applied and known caveats, counted by reason

| Section | What it shows |
| --- | --- |
| 1 | Checks from the latest `01_build_analytics` run |
| 2 | Edge cases: Analytics values vs values calculated from the source files |
| 3 | Exclusions and caveats |

In [0]:
from pyspark.sql import functions as F

CATALOG, ANALYTICS, GOLD, DQ = "tuklas_dev", "05_analytics", "04_gold", "06_data_quality"
def t(schema, name): return f"`{CATALOG}`.`{schema}`.`{name}`"

TOL = 0.01   # tolerance for % and per-1,000 values (Gold stores money rounded to whole thousand PhP)

## 1. Checks from the latest run

`01_build_analytics` saves its checks to `dq_results`: Gold input checks, checks on each Analytics table before it is written, and cross-table checks against Gold. This shows the latest run: a summary by severity and status, then every check with failures first.

In [0]:
dq = spark.table(t(DQ, "dq_results")).filter("notebook_name = '01_build_analytics'")
latest = dq.agg(F.max("checked_at")).collect()[0][0]
run_id = dq.filter(F.col("checked_at") == latest).select("run_id").first()[0]
run = dq.filter(F.col("run_id") == run_id)

print(f"Latest run: {run_id} ({latest})")
display(run.groupBy("severity", "status").count().orderBy("severity", "status"))
display(run.select("check_id", "table_name", "severity", "status", "failed_rows", "notes")
           .orderBy(F.col("status").desc(), "check_id"))

## 2. Edge cases: Analytics vs source files

Expected values were calculated directly from DOT *Regional Distribution of Overnight Travelers* (2019, 2022, 2023), PSA Tables 3.72, 3.94 and 3.95, and PSA DS_18, applying DL-011 (Lucena in Quezon), DL-013 (Boracay/Aklan), DL-014 (Clark/Pampanga), DL-019 (`-` values) and DL-021 (population).

| Unit | Edge case |
| --- | --- |
| Aklan | Boracay; not comparable to 2019 from 2021 |
| Pampanga | Clark Freeport Zone added back from 2023 |
| Quezon | Lucena inside Quezon |
| City of Baguio | Separately reported HUC |
| Abra | Missing DOT value (2023 `not_reported`) |
| Basilan | No DOT data |
| Aurora, Camiguin | Units shown in the presentation |

Traveler and population counts must match exactly; % and per-1,000 values within 0.01. NULL on both sides counts as a match. A&F growth is checked against constant prices and A&F share against current prices, so a match also confirms the price basis.

In [0]:
# (code, unit, trav_2019, trav_2022, trav_2023, pop_2023, intensity_2023, RI_2022, RI_2023,
#  tourism_change_pct, afs_gva_growth_pct, afs_share_of_gdp_pct, pattern)
EXPECTED = [
    ("0600400000", "Aklan",          2251966, 1759592, 2200138,  620744, 3544.36, None,   None,   None,   -35.05,  3.24, None),
    ("0305400000", "Pampanga",        910666, 1124190, 1655887, 2465728,  671.56, 123.45, 181.83,  81.83,   6.50,  2.65, "tourism_up_economy_up"),
    ("0405600000", "Quezon",         1301486, 1186369, 1264181, 2328096,  543.01,  91.15,  97.13,  -2.87,  50.57,  4.05, "tourism_down_economy_up"),
    ("1430300000", "City of Baguio", 1536458, 1042309, 1309930,  371374, 3527.25,  67.84,  85.26, -14.74,   9.35,  9.96, "tourism_down_economy_up"),
    ("1400100000", "Abra",              2948,    1511,    None,  250399,    None,  51.26,   None,   None,  57.16,  0.61, None),
    ("1900700000", "Basilan",           None,    None,    None,  413710,    None,   None,   None,   None,   0.84,  0.10, None),
    ("0307700000", "Aurora",          273906,  515739,  736339,  234084, 3145.62, 188.29, 268.83, 168.83, -25.05,  1.27, "tourism_up_economy_down"),
    ("1001800000", "Camiguin",        804745,  127859,  259010,   94373, 2744.53,  15.89,  32.19, -67.81, -15.35, 11.80, "tourism_down_economy_down"),
]
exp = spark.createDataFrame(EXPECTED, schema=(
    "province_psgc STRING, unit_name STRING, travelers_2019 BIGINT, travelers_2022 BIGINT, travelers_2023 BIGINT, "
    "population_2023 BIGINT, intensity_2023 DOUBLE, recovery_index_2022 DOUBLE, recovery_index_2023 DOUBLE, "
    "tourism_change_pct DOUBLE, afs_gva_growth_pct DOUBLE, afs_share_of_gdp_pct DOUBLE, tourism_economy_pattern STRING"))

# Actual values from the Analytics tables
act_activity = (spark.table(t(ANALYTICS, "tourism_activity")).filter("year = 2023")
                .select("province_psgc",
                        F.col("population_total").alias("a_population_2023"),
                        F.col("tourism_intensity_per_1000").alias("a_intensity_2023")))
act_recovery = (spark.table(t(ANALYTICS, "tourism_recovery"))
                .groupBy("province_psgc").pivot("comparison_year", [2022, 2023])
                .agg(F.first("travelers_comparison_year").alias("trav"),
                     F.first("recovery_index").alias("ri"),
                     F.first("travelers_2019").alias("t19")))
act_economy = (spark.table(t(ANALYTICS, "tourism_economy"))
               .select("province_psgc",
                       F.col("tourism_change_pct").alias("a_tourism_change_pct"),
                       F.col("afs_gva_growth_pct").alias("a_afs_gva_growth_pct"),
                       F.col("afs_share_of_gdp_pct").alias("a_afs_share_of_gdp_pct"),
                       F.col("tourism_economy_pattern").alias("a_pattern")))

cmp = (exp.join(act_activity, "province_psgc", "left")
          .join(act_recovery, "province_psgc", "left")
          .join(act_economy, "province_psgc", "left"))

# expected column -> actual column
pairs = [("travelers_2019", "2022_t19"), ("travelers_2022", "2022_trav"), ("travelers_2023", "2023_trav"),
         ("population_2023", "a_population_2023"), ("intensity_2023", "a_intensity_2023"),
         ("recovery_index_2022", "2022_ri"), ("recovery_index_2023", "2023_ri"),
         ("tourism_change_pct", "a_tourism_change_pct"), ("afs_gva_growth_pct", "a_afs_gva_growth_pct"),
         ("afs_share_of_gdp_pct", "a_afs_share_of_gdp_pct")]

long = None
for e, a in pairs:
    part = cmp.select("unit_name", F.lit(e).alias("metric"),
                      F.col(e).cast("double").alias("expected"),
                      F.col(f"`{a}`").cast("double").alias("actual"))
    long = part if long is None else long.unionByName(part)

# Counts must match exactly; % and per-1,000 values within TOL
long = (long
    .withColumn("diff", F.when(F.col("expected").isNull() & F.col("actual").isNull(), F.lit(0.0))
                         .otherwise(F.abs(F.col("expected") - F.col("actual"))))
    .withColumn("limit", F.when(F.col("metric").startswith("travelers") | (F.col("metric") == "population_2023"), F.lit(0.0))
                          .otherwise(F.lit(TOL)))
    .withColumn("ok", F.col("diff").isNotNull() & (F.col("diff") <= F.col("limit")))
    .drop("limit"))

display(long.orderBy("ok", "unit_name", "metric"))
display(cmp.select("unit_name", "tourism_economy_pattern", "a_pattern",
                   F.col("tourism_economy_pattern").eqNullSafe(F.col("a_pattern")).alias("ok")))
print(f"Edge-case values not matching: {long.filter('NOT ok').count()} of {long.count()}")

## 3. Exclusions and caveats

Units are kept in the Analytics tables even when a metric does not apply, with the reason recorded. This section counts those exclusions by reason, shows the recovery classes and tourism–economy patterns, and lists the caveat fields available in Gold.

In [0]:
rec = spark.table(t(ANALYTICS, "tourism_recovery"))
display(rec.filter("NOT recovery_eligible").groupBy("comparison_year", "recovery_exclusion_reason").count()
           .orderBy("comparison_year", "recovery_exclusion_reason"))
display(rec.groupBy("comparison_year", "recovery_class").count().orderBy("comparison_year", "recovery_class"))

eco = spark.table(t(ANALYTICS, "tourism_economy"))
display(eco.filter("NOT alignment_eligible").groupBy("alignment_exclusion_reason").count())
display(eco.filter("alignment_eligible").groupBy("tourism_economy_pattern").count().orderBy("tourism_economy_pattern"))

gold = spark.table(t(GOLD, "fact_province_year"))
caveats = [c for c in ["foreign_includes_overseas_filipinos", "derived_values", "per_capita_recomputed"] if c in gold.columns]
print("Gold caveat fields present:", caveats)

seg = spark.table(t(ANALYTICS, "traveler_segment_recovery"))
display(seg.groupBy("foreign_segment_supported", "overseas_filipino_segment_supported").count())